# Estacionalidad histórica de los incendios

**Objetivo:** Frecuencia mensual, mapa de calor y resumen histórico.

Este cuaderno contiene el código completo del experimento `analyze_seasonality.py`,
dividido en etapas y acompañado de explicaciones. No necesita ZIP, clonar un
repositorio ni ejecutar un script oculto. Ejecuta las celdas en orden.

**Recorrido:** entorno → carga e inspección → preparación → modelo o análisis →
evaluación → gráficos → descarga y colaboración.

**Entradas:** `incendios_coordenadas_gee.csv`.

Los datos no se incluyen. Se conservan los valores predeterminados del protocolo histórico.
Las salidas se generarán al ejecutar; las figuras históricas no están incrustadas.


## 1. Entorno y parámetros
Modifica los parámetros aquí para que una colaboración sea fácil de revisar. Los valores originales siguen seleccionados. Cada ejecución crea una carpeta nueva.

In [ ]:
# Instala únicamente las bibliotecas que falten en el entorno.
import importlib.util
import importlib.metadata
import subprocess
import sys
import os
import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display, Image

DEPENDENCIAS = {'numpy': 'numpy', 'pandas': 'pandas', 'matplotlib': 'matplotlib', 'seaborn': 'seaborn'}
faltantes = [paquete for modulo, paquete in DEPENDENCIAS.items()
             if importlib.util.find_spec(modulo) is None]
if faltantes:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *faltantes], check=True)

CONFIG = dict(seed=42, rf_trees=100, rf_depth=15, epochs=50,
              learning_rate=0.002, batch_size=256)
EXPERIMENTO = 'estacionalidad'
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
DATA_DIR = Path(globals().get('DATA_DIR_OVERRIDE', 'data')).resolve()
OUTPUT_DIR = Path(globals().get('OUTPUT_DIR_OVERRIDE', Path('results') / EXPERIMENTO / RUN_ID)).resolve()
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLBACKEND', 'Agg')
print('Parámetros:', CONFIG)
print('Salidas:', OUTPUT_DIR)

def mostrar_figura(ruta):
    # Conserva la figura en disco y también la muestra en el cuaderno.
    if str(ruta).lower().endswith('.png'):
        display(Image(filename=str(ruta)))


## 2. Carga e inspección de los datos
En Colab, la siguiente celda ofrece el selector si faltan archivos. Carga solo los CSV indicados, conservando sus nombres. En un entorno local, colócalos en `data/`. Para el dataset realista usa la copia de la subcarpeta `resultados` original.

In [ ]:
import pandas as pd

ARCHIVOS_REQUERIDOS = ['incendios_coordenadas_gee.csv']
def localizar(nombre):
    candidatos = [DATA_DIR / nombre, DATA_DIR / 'resultados' / nombre]
    return next((p for p in candidatos if p.is_file()), candidatos[0])

pendientes = [n for n in ARCHIVOS_REQUERIDOS if not localizar(n).is_file()]
if pendientes:
    print('Archivos requeridos:', pendientes)
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError('Coloca los CSV en ' + str(DATA_DIR))
    cargados = files.upload()
    for nombre, contenido in cargados.items():
        if Path(nombre).name != nombre or nombre not in ARCHIVOS_REQUERIDOS:
            raise ValueError('Archivo inesperado: ' + nombre)
        (DATA_DIR / nombre).write_bytes(contenido)
    del cargados

DATA_FILES = {n: localizar(n) for n in ARCHIVOS_REQUERIDOS}
if any(not p.is_file() for p in DATA_FILES.values()):
    raise FileNotFoundError('Faltan CSV; repite esta celda y carga los indicados.')

# Inspección visible: dimensiones, tipos, valores faltantes y balance de clases.
for nombre, ruta in DATA_FILES.items():
    tabla = pd.read_csv(ruta)
    print('\n', nombre, '| filas y columnas:', tabla.shape)
    display(tabla.head())
    display(pd.DataFrame({'tipo': tabla.dtypes.astype(str), 'nulos': tabla.isna().sum()}))
    if 'clase' in tabla.columns:
        display(tabla['clase'].value_counts(dropna=False).rename('filas').to_frame())

REGISTRO = dict(experimento=EXPERIMENTO, parametros=CONFIG, run_id=RUN_ID,
    input_sha256={n: hashlib.sha256(p.read_bytes()).hexdigest() for n, p in DATA_FILES.items()},
    versiones={p: importlib.metadata.version(p) for p in DEPENDENCIAS.values()},
    estado='iniciado')
(OUTPUT_DIR / 'ejecucion.json').write_text(json.dumps(REGISTRO, ensure_ascii=False, indent=2), encoding='utf-8')


## 3. Bibliotecas, rutas y reproducibilidad
Estas son las importaciones, rutas de entrada/salida y semillas del experimento. Los archivos generados se guardan en la carpeta de esta ejecución.

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial', 'Helvetica']
plt.rcParams['font.size'] = 9
plt.rcParams['axes.labelsize'] = 9.5
plt.rcParams['axes.titlesize'] = 10
plt.rcParams['xtick.labelsize'] = 8.5
plt.rcParams['ytick.labelsize'] = 8.5
plt.rcParams['legend.fontsize'] = 8.5
plt.rcParams['savefig.bbox'] = 'tight'
plt.rcParams['savefig.pad_inches'] = 0.05
BASE_DIR = str(OUTPUT_DIR)
OUT_DIR = str(OUTPUT_DIR)
FILE_GEE_CSV = str(DATA_FILES['incendios_coordenadas_gee.csv'])


## 4. Desarrollo del experimento
Las siguientes celdas contienen el cuerpo del análisis original, con comentarios y código editable. Puedes inspeccionar las variables entre pasos. Ejecuta nuevamente desde la carga de datos si cambias una transformación previa.

### Etapa 1: Carga y preparación de datos
Comprueba las columnas, las filas descartadas y el significado de la etiqueta antes de entrenar. El código conserva la selección de variables del experimento original.

In [ ]:
print("="*75)
print("  ANÁLISIS DE ESTACIONALIDAD HISTÓRICA - EXCLUSIVO SIERRA ECUATORIANA")
print("="*75)

if not os.path.exists(FILE_GEE_CSV):
    print(f"[ERROR] No se encuentra el archivo: {FILE_GEE_CSV}")
    raise ValueError("Revisa el archivo y las columnas indicadas en la salida anterior.")
df = pd.read_csv(FILE_GEE_CSV)
print(f"[INFO] Dataset total nacional cargado: {len(df)} registros.")

# Lista exacta de los nombres de provincia de la Sierra presentes en el CSV
raw_sierra_provinces = [
    "'Pichincha'", "'Azuay'", "'Imbabura'", "'Loja'", "'Tungurahua'", 
    "'Chimborazo'", "'Caar'", "'Cañar'", "'Cotopaxi'", "'Carchi'", "'Bolvar'", "'Bolívar'"
]

def es_sierra(prov_val):
    if not isinstance(prov_val, str):
        return False
    p_clean = prov_val.strip().strip("'").lower()
    return any(k in p_clean for k in ['pichincha', 'azuay', 'imbabura', 'loja', 'tungurahua', 'chimborazo', 'ca', 'cotopaxi', 'carchi', 'bol']) and not any(k in p_clean for k in ['pastaza', 'orellana', 'galapagos', 'esmeraldas', 'los rios', 'el oro', 'guayas', 'manabi', 'santa elena', 'santo domingo', 'morona', 'zamora', 'napo', 'sucumbi'])

df_sierra = df[df['provincia'].apply(es_sierra)].copy()
print(f"[INFO] Registros filtrados EXCLUSIVAMENTE para la SIERRA ({len(df_sierra)} presencias).")
print("       Provincias incluidas:", df_sierra['provincia'].unique().tolist())

# Parsear fechas
df_sierra['fecha_parsed'] = pd.to_datetime(df_sierra['date'], errors='coerce')


### Etapa 2: Carga y preparación de datos
Comprueba las columnas, las filas descartadas y el significado de la etiqueta antes de entrenar. El código conserva la selección de variables del experimento original.

In [ ]:
df_valid = df_sierra.dropna(subset=['fecha_parsed']).copy()

df_valid['anio'] = df_valid['fecha_parsed'].dt.year
df_valid['mes_num'] = df_valid['fecha_parsed'].dt.month

nombres_meses = ['Enero', 'Febrero', 'Marzo', 'Abril', 'Mayo', 'Junio', 
                 'Julio', 'Agosto', 'Septiembre', 'Octubre', 'Noviembre', 'Diciembre']
nombres_meses_abrev = ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 
                       'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic']

df_valid['mes_nombre'] = df_valid['mes_num'].apply(lambda x: nombres_meses[x-1])

# 2. Conteo por mes (1-12) en la Sierra
conteo_mensual = df_valid['mes_num'].value_counts().sort_index()
total_incendios_sierra = len(df_valid)
num_anios = df_valid['anio'].nunique()

conteo_mensual = conteo_mensual.reindex(range(1, 13), fill_value=0)
porcentajes_mensuales = (conteo_mensual / total_incendios_sierra) * 100
promedio_anual_mensual = conteo_mensual / num_anios

# Tabla resumen Sierra
df_resumen = pd.DataFrame({
    'Mes': nombres_meses,
    'Número de incendios': conteo_mensual.values,
    'Porcentaje del total': porcentajes_mensuales.values,
    'Promedio anual': promedio_anual_mensual.values
})


### Etapa 3: Representación y exportación
Lee los ejes y el conjunto de datos utilizado. Las figuras PNG se muestran al guardarse; al final también puedes inspeccionar todas las salidas y descargarlas.

In [ ]:

print("\n" + "="*65)
print(f"  TABLA RESUMEN SIERRA ECUATORIANA ({num_anios} AÑOS HISTÓRICOS)")
print("="*65)
print(df_resumen.to_string(index=False, formatters={
    'Número de incendios': '{:,}'.format,
    'Porcentaje del total': '{:.2f}%'.format,
    'Promedio anual': '{:.1f}'.format
}))

csv_resumen_path = os.path.join(OUT_DIR, "seasonality_summary_table.csv")
df_resumen.to_csv(csv_resumen_path, index=False)
print(f"\n[OK] Tabla Sierra guardada en: {csv_resumen_path}")

# Meses pico y comprobación de agosto
mes_max = df_resumen.loc[df_resumen['Número de incendios'].idxmax()]['Mes']
pico_top3 = df_resumen.sort_values(by='Número de incendios', ascending=False).head(3)['Mes'].tolist()
incendios_agosto = df_resumen.loc[df_resumen['Mes'] == 'Agosto', 'Número de incendios'].values[0]
pct_agosto = df_resumen.loc[df_resumen['Mes'] == 'Agosto', 'Porcentaje del total'].values[0]

print(f"\n[HALLAZGO SIERRA] Mes de mayor frecuencia histórica: {mes_max}")
print(f"[HALLAZGO SIERRA] Top 3 meses con mayor frecuencia: {', '.join(pico_top3)}")
print(f"[HALLAZGO SIERRA] Agosto: {incendios_agosto:,} incendios ({pct_agosto:.2f}% del total de la Sierra).")

# =========================================================================
# FIGURA A: BARRAS DEL NÚMERO TOTAL DE INCENDIOS POR MES (SIERRA ECUATORIANA)
# =========================================================================
fig, ax = plt.subplots(figsize=(6.5, 4.0), dpi=600)


### Etapa 4: Representación y exportación
Lee los ejes y el conjunto de datos utilizado. Las figuras PNG se muestran al guardarse; al final también puedes inspeccionar todas las salidas y descargarlas.

In [ ]:
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

bars = ax.bar(nombres_meses_abrev, conteo_mensual.values, color='#d95f0e', edgecolor='#a63603', lw=0.8, width=0.65)

# Destacar agosto y septiembre en rojo
bars[7].set_color('#99000d') # Agosto
bars[8].set_color('#cc4c02') # Septiembre

ax.set_ylabel('Total Number of Wildfires (Ecuadorian Andes)', fontweight='bold')
ax.set_xlabel('Month', fontweight='bold')
ax.set_title('Monthly Wildfire Frequency Distribution in the Ecuadorian Andes', fontweight='bold', pad=10)
ax.grid(True, linestyle=':', alpha=0.5, color='#cccccc', axis='y')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

for bar in bars:
    height = bar.get_height()
    if height > 0:
        ax.annotate(f'{int(height):,}',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3),
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=7.5, fontweight='bold')

plt.tight_layout()


### Etapa 5: Representación y exportación
Lee los ejes y el conjunto de datos utilizado. Las figuras PNG se muestran al guardarse; al final también puedes inspeccionar todas las salidas y descargarlas.

In [ ]:
fn_figa_png = os.path.join(OUT_DIR, "seasonality_monthly_fires.png")
fn_figa_pdf = os.path.join(OUT_DIR, "seasonality_monthly_fires.pdf")
plt.savefig(fn_figa_png, dpi=600, facecolor='white')
mostrar_figura(fn_figa_png)
plt.savefig(fn_figa_pdf, facecolor='white')
mostrar_figura(fn_figa_pdf)
plt.close()
print(f"\n[OK] FIGURA A SIERRA guardada: {fn_figa_png} | {fn_figa_pdf}")

# =========================================================================
# FIGURA B: HEATMAP AÑO x MES (SIERRA ECUATORIANA)
# =========================================================================
pivot_anio_mes = df_valid.pivot_table(index='anio', columns='mes_num', values='id', aggfunc='count', fill_value=0)
pivot_anio_mes = pivot_anio_mes.reindex(columns=range(1, 13), fill_value=0)
pivot_anio_mes.columns = nombres_meses_abrev
pivot_anio_mes = pivot_anio_mes[pivot_anio_mes.sum(axis=1) > 10].sort_index(ascending=False)

fig, ax = plt.subplots(figsize=(7.5, 5.0), dpi=600)
fig.patch.set_facecolor('white')

sns.heatmap(pivot_anio_mes, annot=True, fmt='d', cmap='YlOrRd', cbar_kws={'label': 'Number of Wildfires'},
            linewidths=0.5, linecolor='#ffffff', ax=ax, annot_kws={"size": 7.5, "weight": "normal"})

ax.set_title('Year-Month Wildfire Occurrence Heatmap (Ecuadorian Andes)', fontweight='bold', pad=10)
ax.set_xlabel('Month', fontweight='bold')
ax.set_ylabel('Year', fontweight='bold')


### Etapa 6: Representación y exportación
Lee los ejes y el conjunto de datos utilizado. Las figuras PNG se muestran al guardarse; al final también puedes inspeccionar todas las salidas y descargarlas.

In [ ]:

plt.tight_layout()
fn_figb_png = os.path.join(OUT_DIR, "seasonality_heatmap_year_month.png")
fn_figb_pdf = os.path.join(OUT_DIR, "seasonality_heatmap_year_month.pdf")
plt.savefig(fn_figb_png, dpi=600, facecolor='white')
mostrar_figura(fn_figb_png)
plt.savefig(fn_figb_pdf, facecolor='white')
mostrar_figura(fn_figb_pdf)
plt.close()
print(f"[OK] FIGURA B SIERRA guardada: {fn_figb_png} | {fn_figb_pdf}")

print("\n" + "="*75)
print("  ANÁLISIS DE ESTACIONALIDAD EXCLUSIVO DE LA SIERRA COMPLETADO")
print("="*75)


## 5. Inspeccionar y guardar resultados
Las tablas se muestran debajo. El registro identifica parámetros, versiones y hashes. Esta celda marca el registro como completado: ejecútala solo cuando hayan terminado todas las etapas anteriores.

In [ ]:
for ruta in sorted(OUTPUT_DIR.glob('*.csv')):
    print(ruta.name)
    display(pd.read_csv(ruta).head(20))
REGISTRO['estado'] = 'completado'
(OUTPUT_DIR / 'ejecucion.json').write_text(json.dumps(REGISTRO, ensure_ascii=False, indent=2), encoding='utf-8')
print('Salidas:', [p.name for p in sorted(OUTPUT_DIR.iterdir())])


In [ ]:
# Descarga las salidas antes de cerrar una sesión de Colab.
import shutil
paquete = shutil.make_archive(str(OUTPUT_DIR.parent / (EXPERIMENTO + '-' + RUN_ID)), 'zip', OUTPUT_DIR)
try:
    from google.colab import files
except ImportError:
    print('ZIP guardado en:', paquete)
else:
    files.download(paquete)


## 6. Proponer una colaboración

1. Plantea una pregunta concreta: por ejemplo, el efecto de un hiperparámetro o de una variable.
2. Conserva una ejecución con la configuración original y luego cambia una sola decisión.
3. Registra el hash del dataset, la partición, la semilla y todas las métricas relevantes.
4. En GitHub, abre un Issue para discutir la propuesta. Trabaja en un fork o en una rama
   con permisos y envía un pull request que explique el cambio y su evidencia.
5. Antes del commit, borra las salidas del cuaderno y revisa que no incluyan datos privados.
   Adjunta resultados únicamente cuando tengas permiso para compartirlos.

Edita la siguiente celda Markdown para documentar tu propuesta. No se envía nada automáticamente.

### Registro de la propuesta

- **Autor y objetivo:**
- **Hipótesis:**
- **Parámetro o código modificado:**
- **Dataset y hash:**
- **Semilla y partición:**
- **Resultado original y resultado de la variante:**
- **Limitaciones y conclusión:**
